# IBM Applied Data Science Capstone
## Falcon 9 landing prediction - data wrangling

**Learner:** Djessi Jorge  
**Completed:** 4 August 2026

The objective is to audit the API snapshot, create the binary landing target and publish
model-ready numeric features without discarding legitimate no-pad records.

In [1]:
import numpy as np
import pandas as pd

api_data = pd.read_csv("dataset_part_1.csv")
print(f"API-stage input shape: {api_data.shape}")
api_data.head()

API-stage input shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [2]:
audit = pd.DataFrame(
    {
        "dtype": api_data.dtypes.astype(str),
        "missing": api_data.isna().sum(),
        "missing_pct": (100 * api_data.isna().mean()).round(2),
        "unique": api_data.nunique(dropna=False),
    }
)
audit

,dtype,missing,missing_pct,unique
FlightNumber,int64,0,0.00,90
Date,str,0,0.00,90
BoosterVersion,str,0,0.00,1
PayloadMass,float64,0,0.00,68
Orbit,str,0,0.00,11
LaunchSite,str,0,0.00,3
Outcome,str,0,0.00,8
Flights,int64,0,0.00,6
GridFins,bool,0,0.00,2
Reused,bool,0,0.00,2


In [3]:
api_data["PayloadMass"] = api_data["PayloadMass"].fillna(
    api_data["PayloadMass"].mean()
)
api_data["LaunchSite"] = api_data["LaunchSite"].replace(
    {"CCSFS SLC 40": "CCAFS SLC 40", "CCAFS LC-40": "CCAFS SLC 40"}
)

unsuccessful = {
    "False ASDS",
    "False Ocean",
    "False RTLS",
    "None ASDS",
    "None None",
}
api_data["Class"] = (~api_data["Outcome"].isin(unsuccessful)).astype(int)

# The supplied part-2 course snapshot freezes dynamic API fields such as core reuse
# counts and payload updates. Keeping that frozen snapshot makes the published model
# results reproducible even as live SpaceX records change.
data = pd.read_csv("dataset_part_2.csv")
assert api_data["Class"].equals(data["Class"])

target_summary = data["Class"].value_counts().sort_index().rename(
    index={0: "Failure", 1: "Success"}
)
target_summary.to_frame("launches")

,launches
Class,
Failure,30
Success,60


In [4]:
site_summary = (
    data.groupby("LaunchSite")["Class"]
    .agg(launches="size", successes="sum", success_rate="mean")
    .assign(success_rate=lambda x: (100 * x["success_rate"]).round(1))
    .sort_values("success_rate", ascending=False)
)
site_summary

,launches,successes,success_rate
LaunchSite,,,
KSC LC 39A,22,17,77.3
VAFB SLC 4E,13,10,76.9
CCAFS SLC 40,55,33,60.0


In [5]:
data.to_csv("dataset_part_2.csv", index=False)

feature_columns = [
    "FlightNumber", "PayloadMass", "Flights", "GridFins", "Reused", "Legs",
    "Block", "ReusedCount", "Orbit", "LaunchSite", "LandingPad", "Serial"
]
categorical_columns = ["Orbit", "LaunchSite", "LandingPad", "Serial"]

features = pd.get_dummies(
    data[feature_columns], columns=categorical_columns, dtype=int
).astype(float)
features.to_csv("dataset_part_3.csv", index=False)

print(f"dataset_part_2.csv: {data.shape}")
print(f"dataset_part_3.csv: {features.shape}")
features.head()

dataset_part_2.csv: (90, 18)
dataset_part_3.csv: (90, 80)


,FlightNumber,PayloadMass,Flights,GridFins,Reused,Legs,Block,ReusedCount,Orbit_ES-L1,Orbit_GEO,Orbit_GTO,Orbit_HEO,Orbit_ISS,Orbit_LEO,Orbit_MEO,Orbit_PO,Orbit_SO,Orbit_SSO,Orbit_VLEO,LaunchSite_CCAFS SLC 40,LaunchSite_KSC LC 39A,LaunchSite_VAFB SLC 4E,LandingPad_5e9e3032383ecb267a34e7c7,LandingPad_5e9e3032383ecb554034e7c9,LandingPad_5e9e3032383ecb6bb234e7ca,LandingPad_5e9e3032383ecb761634e7cb,LandingPad_5e9e3033383ecbb9e534e7cc,Serial_B0003,Serial_B0005,Serial_B0007,Serial_B1003,Serial_B1004,Serial_B1005,Serial_B1006,Serial_B1007,Serial_B1008,Serial_B1010,Serial_B1011,Serial_B1012,Serial_B1013,Serial_B1015,Serial_B1016,Serial_B1017,Serial_B1018,Serial_B1019,Serial_B1020,Serial_B1021,Serial_B1022,Serial_B1023,Serial_B1025,Serial_B1026,Serial_B1028,Serial_B1029,Serial_B1030,Serial_B1031,Serial_B1032,Serial_B1034,Serial_B1035,Serial_B1036,Serial_B1037,Serial_B1038,Serial_B1039,Serial_B1040,Serial_B1041,Serial_B1042,Serial_B1043,Serial_B1044,Serial_B1045,Serial_B1046,Serial_B1047,Serial_B1048,Serial_B1049,Serial_B1050,Serial_B1051,Serial_B1054,Serial_B1056,Serial_B1058,Serial_B1059,Serial_B1060,Serial_B1062
0,1.0,6104.959412,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,525.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,677.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4.0,500.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.0,3170.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
assert data.shape == (90, 18)
assert features.shape == (90, 80)
assert data["Class"].sum() == 60
assert data["Class"].mean() == 2 / 3
assert features.isna().sum().sum() == 0

print("Validation passed: 90 rows, 80 model features, 60 successful landings (66.67%).")

Validation passed: 90 rows, 80 model features, 60 successful landings (66.67%).


### Result

The target contains **60 successful** and **30 unsuccessful** landings, giving a historical
success rate of **66.67%**. One-hot encoding produces 80 numeric model features across the
same 90 launch records.